<a href="https://colab.research.google.com/github/UNOKOR/graduationProject/blob/main/graduationproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 설정

In [ ]:
CFG = dict(
    exp_name = "None",

    # 학습
    epochs = 200,
    batch_size = 256,
    num_workers = 2,
    label_smoothing = 0.1,
    warmup_epochs = 2,
    seed = 43,

    # Muon (은닉층 conv/linear 가중치)
    use_muon=True,           # False면 모든 파라미터를 보조 옵티마이저(SGD 등)로 학습 (Muon 없는 baseline)
    muon_lr=0.02,
    muon_momentum=0.95,
    muon_weight_decay=5e-3,  # decoupled
    ns_steps=5,              # Newton-Schulz 반복 횟수

    # 보조 옵티마이저 (첫 conv, fc, BN, bias)
    aux_optimizer="sgd",     # "sgd" 또는 "adamw"
    aux_lr=0.05,             # adamw를 쓰면 1e-3 ~ 3e-3 권장
    aux_momentum=0.9,
    aux_weight_decay=5e-4,

    # SAM
    use_sam=True,            # False면 SAM 없이 학습
    sam_mode="proposed",     # "proposed": ε_l = ρ·‖W_l‖_op·Orth(G_l)  (이 노트북의 변형)
                                 # "spectral": ε_l = ρ·Orth(G_l) (Spectral SAM 논문) | "l2": 원래 SAM | "asam": ASAM
    sam_rho=1e-3,            # proposed: 층 크기 대비 비율 (‖ε_l‖_op / ‖W_l‖_op). 5e-4 ~ 5e-3 를 짧게 스윕 권장
                             # spectral: 5e-4 ~ 2e-3 | l2: 0.05 | asam: 0.5 ~ 2.0
    sam_rho_vec=0.0,         # spectral 모드에서 BN, bias(1D)에 줄 L2 교란 반경 (논문 보고: 0이 무난)
    sam_power_iters=1,       # ‖W_l‖_op 추정용 power iteration 횟수 (스텝마다, 이전 벡터를 이어서 사용)

    # 기타
    use_amp=True,
    step_log_every=20,       # N 스텝마다 L(w), L(w+ε), 업데이트 후 L 기록 (0이면 끔). 기록할 때만 forward 1회 추가
    train_subset=None,       # 빠른 확인용: 예) 5000 (None이면 전체 50,000장)
    save_to_drive=True,      # Drive의 MyDrive/muon_sam_cifar100/ 에 결과와 데이터 캐시 저장 (런타임이 끊겨도 유지)
    resume=True,             # 같은 exp_name 폴더에 last.pt가 있으면 그 epoch부터 이어서 학습
)
CFG

## 2. 환경 준비

In [ ]:
import math, os, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요.")

# T4(sm75)는 bf16 연산이 느리므로 fp16 + GradScaler, Ampere 이상은 bf16
use_bf16 = device == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
amp_enabled = CFG["use_amp"] and device == "cuda"
print("AMP:", amp_enabled, amp_dtype if amp_enabled else "")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])
torch.backends.cudnn.benchmark = True

import json, shutil

if CFG["save_to_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/muon_sam_cifar100"
else:
    ROOT = "./muon_sam_cifar100"
RUNS_DIR = os.path.join(ROOT, "runs")
os.makedirs(RUNS_DIR, exist_ok=True)

if not CFG["exp_name"]:
    opt_name = "muon" if CFG["use_muon"] else CFG["aux_optimizer"]
    sam_tag = {"proposed": "proposed", "spectral": "specsam", "l2": "sam", "asam": "asam"}[CFG["sam_mode"]]
    sam_name = f"+{sam_tag}_rho{CFG['sam_rho']:g}" if CFG["use_sam"] else ""
    CFG["exp_name"] = f"{opt_name}{sam_name}_e{CFG['epochs']}"
CFG["save_dir"] = os.path.join(RUNS_DIR, CFG["exp_name"])
os.makedirs(CFG["save_dir"], exist_ok=True)
with open(os.path.join(CFG["save_dir"], "config.json"), "w") as f:
    json.dump(CFG, f, indent=2, ensure_ascii=False)
print("실험:", CFG["exp_name"], "| 저장 위치:", CFG["save_dir"])
print("기존 실험:", sorted(os.listdir(RUNS_DIR)))

## 3. 데이터 (CIFAR-100)

In [ ]:
MEAN, STD = (0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)
train_tf = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

# 처음 한 번만 인터넷에서 받아 Drive(ROOT/data)에 캐시하고, 이후에는 Drive에서 로컬로 복사해 사용
LOCAL_DATA = "./data"
local_dir = os.path.join(LOCAL_DATA, "cifar-100-python")
cache_dir = os.path.join(ROOT, "data", "cifar-100-python")
if not os.path.exists(local_dir) and os.path.exists(cache_dir):
    shutil.copytree(cache_dir, local_dir)
    print("Drive 캐시에서 CIFAR-100을 불러왔습니다.")

train_set = torchvision.datasets.CIFAR100(LOCAL_DATA, train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR100(LOCAL_DATA, train=False, download=True, transform=test_tf)

if not os.path.exists(cache_dir):
    shutil.copytree(local_dir, cache_dir)
    print("CIFAR-100을 Drive에 캐시했습니다. 다음 실험부터는 다시 받지 않습니다.")
if CFG["train_subset"]:
    train_set = torch.utils.data.Subset(train_set, range(CFG["train_subset"]))

loader_kw = dict(num_workers=CFG["num_workers"], pin_memory=(device == "cuda"),
                 persistent_workers=CFG["num_workers"] > 0)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=CFG["batch_size"], shuffle=True, drop_last=True, **loader_kw)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512, shuffle=False, **loader_kw)
print(f"train {len(train_set)} | test {len(test_set)} | steps/epoch {len(train_loader)}")

## 4. 모델: ResNet-18 (CIFAR 변형)

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, planes, 1, stride, bias=False), nn.BatchNorm2d(planes))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet(nn.Module):
    def __init__(self, num_blocks=(2, 2, 2, 2), num_classes=100):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)   # CIFAR용 stem (maxpool 없음)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], 1)
        self.layer2 = self._make_layer(128, num_blocks[1], 2)
        self.layer3 = self._make_layer(256, num_blocks[2], 2)
        self.layer4 = self._make_layer(512, num_blocks[3], 2)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, n, stride):
        layers = []
        for s in [stride] + [1] * (n - 1):
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)

def resnet18(num_classes=100):
    return ResNet((2, 2, 2, 2), num_classes)

model = resnet18().to(device).to(memory_format=torch.channels_last)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

params: 11.22M


## 5. Muon 옵티마이저


In [ ]:
# Ampere 이상은 bf16, T4 등은 fp32로 Newton-Schulz 계산
NS_DTYPE = torch.bfloat16 if use_bf16 else torch.float32

@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5):
    # G ≈ U S V^T 일 때 U V^T 를 근사 (5차 Newton-Schulz)
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.to(NS_DTYPE)
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.lerp_(g, 1 - mom)
                g = g.lerp(buf, mom) if group["nesterov"] else buf
                g2d = g.reshape(g.size(0), -1)
                update = zeropower_via_newtonschulz5(g2d, group["ns_steps"])
                update *= max(1.0, g2d.size(0) / g2d.size(1)) ** 0.5
                if wd:
                    p.mul_(1 - lr * wd)
                p.add_(update.view_as(p).to(p.dtype), alpha=-lr)
        return loss

## 6. 파라미터 분리

In [ ]:
muon_params, aux_params = [], []
for name, p in model.named_parameters():
    if CFG["use_muon"] and p.ndim >= 2 and not name.startswith(("conv1.", "fc.")):
        muon_params.append(p)
    else:
        aux_params.append(p)
print(f"Muon: {len(muon_params)} tensors ({sum(p.numel() for p in muon_params)/1e6:.2f}M) | "
      f"Aux: {len(aux_params)} tensors ({sum(p.numel() for p in aux_params)/1e6:.3f}M)")

if CFG["aux_optimizer"] == "adamw":
    aux_opt = torch.optim.AdamW(aux_params, lr=CFG["aux_lr"], betas=(0.9, 0.95), weight_decay=CFG["aux_weight_decay"])
else:
    aux_opt = torch.optim.SGD(aux_params, lr=CFG["aux_lr"], momentum=CFG["aux_momentum"],
                              nesterov=True, weight_decay=CFG["aux_weight_decay"])
optimizers = [aux_opt]
if muon_params:
    muon_opt = Muon(muon_params, lr=CFG["muon_lr"], momentum=CFG["muon_momentum"],
                    ns_steps=CFG["ns_steps"], weight_decay=CFG["muon_weight_decay"])
    optimizers = [muon_opt, aux_opt]

Muon: 19 tensors (11.16M) | Aux: 43 tensors (0.063M)


## 7. SAM: 층별 스케일

In [ ]:
class SAM:
    def __init__(self, params, mode="proposed", rho=1e-3, rho_vec=0.0, ns_steps=5, power_iters=1):
        assert mode in ("proposed", "spectral", "l2", "asam"), mode
        self.params = [p for p in params if p.requires_grad]
        self.mode, self.rho, self.rho_vec = mode, rho, rho_vec
        self.ns_steps, self.power_iters = ns_steps, power_iters
        self.e_w = {}
        self._u = {}   # 층별 power iteration 벡터 (스텝 사이에 이어서 사용)

    @torch.no_grad()
    def spectral_norm(self, p):
        # ‖W‖_op (가장 큰 특이값)을 power iteration으로 추정. 처음엔 20회, 이후엔 power_iters회
        W = p.detach().reshape(p.size(0), -1).float()
        u = self._u.get(p)
        iters = self.power_iters
        if u is None:
            u = torch.randn(W.size(0), device=W.device)
            u, iters = u / u.norm(), 20
        for _ in range(iters):
            v = W.T @ u
            v = v / (v.norm() + 1e-12)
            u = W @ v
            s = u.norm()
            u = u / (s + 1e-12)
        self._u[p] = u
        return s

    @torch.no_grad()
    def _l2_perturb(self, params, rho, adaptive, grad_scale):
        # 원래 SAM / ASAM: params 전체를 한 벡터로 본 L2 노름 공 위의 교란
        norms = [((p.abs() if adaptive else 1.0) * p.grad).norm(2) for p in params]
        grad_norm = torch.stack(norms).norm(2) / grad_scale
        scale = rho / (grad_norm + 1e-12) / grad_scale
        for p in params:
            e_w = (p.pow(2) if adaptive else 1.0) * p.grad * scale
            p.add_(e_w)
            self.e_w[p] = e_w

    @torch.no_grad()
    def perturb(self, grad_scale=1.0):
        # grad_scale: GradScaler로 스케일된 기울기를 원래 크기로 되돌리기 위한 값 (l2/asam에서만 필요)
        self.e_w = {}
        params = [p for p in self.params if p.grad is not None]
        if not params:
            return False
        if not torch.isfinite(torch.stack([p.grad.norm() for p in params]).sum()):
            return False   # fp16 overflow → 이번 스텝은 교란 생략
        if self.mode in ("l2", "asam"):
            self._l2_perturb(params, self.rho, self.mode == "asam", grad_scale)
            return True
        for p in params:
            if p.ndim < 2:
                continue
            g2d = p.grad.reshape(p.size(0), -1)
            e = zeropower_via_newtonschulz5(g2d, self.ns_steps).float()   # ≈ U V^T, 스펙트럴 노름 ≈ 1
            r = self.rho * self.spectral_norm(p) if self.mode == "proposed" else self.rho
            e_w = (e * r).view_as(p).to(p.dtype)
            p.add_(e_w)
            self.e_w[p] = e_w
        vecs = [p for p in params if p.ndim < 2]
        if self.rho_vec > 0 and vecs:
            self._l2_perturb(vecs, self.rho_vec, False, grad_scale)
        return True

    @torch.no_grad()
    def restore(self):
        for p, e_w in self.e_w.items():
            p.sub_(e_w)
        self.e_w = {}

def set_bn_momentum(model, enable):
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm):
            if enable:
                m.momentum = getattr(m, "_backup_momentum", m.momentum)
            else:
                m._backup_momentum = m.momentum
                m.momentum = 0.0

sam = SAM(model.parameters(), mode=CFG["sam_mode"], rho=CFG["sam_rho"], rho_vec=CFG["sam_rho_vec"],
          ns_steps=CFG["ns_steps"], power_iters=CFG["sam_power_iters"]) if CFG["use_sam"] else None

# 확인용: 층별 ‖W_l‖_op 와 실제 교란 크기 ‖ε_l‖_op (proposed 기준)
if sam is not None and CFG["sam_mode"] == "proposed":
    print(f"{'layer':32s} {'shape':>12s} {'|W|_op':>8s} {'|eps|_op':>9s}")
    for name, p in model.named_parameters():
        if p.ndim >= 2:
            s = sam.spectral_norm(p).item()
            shape = f"{p.size(0)}x{p[0].numel()}"
            print(f"{name:32s} {shape:>12s} {s:8.3f} {CFG['sam_rho'] * s:9.2e}")

## 8. 학습 루프

In [ ]:
epochs = CFG["epochs"]
steps_per_epoch = len(train_loader)
total_steps = epochs * steps_per_epoch
warmup_steps = CFG["warmup_epochs"] * steps_per_epoch

def lr_factor(step):   # linear warmup → cosine decay
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

schedulers = [torch.optim.lr_scheduler.LambdaLR(o, lr_factor) for o in optimizers]
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and amp_dtype == torch.float16)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

def forward_loss(x, y):
    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=amp_enabled):
        out = model(x)
        loss = criterion(out, y)
    return loss, out

def zero_grad():
    for o in optimizers:
        o.zero_grad(set_to_none=True)

def train_step(x, y):
    loss_adv = None
    zero_grad()
    loss, out = forward_loss(x, y)
    scaler.scale(loss).backward()

    if sam is not None:
        # 1) w → w + e(w)
        sam.perturb(grad_scale=scaler.get_scale() if scaler.is_enabled() else 1.0)
        zero_grad()
        # 2) 교란된 지점에서 기울기 계산 (BN 통계는 갱신하지 않음)
        set_bn_momentum(model, False)
        loss_adv, _ = forward_loss(x, y)
        scaler.scale(loss_adv).backward()
        set_bn_momentum(model, True)
        # 3) w 로 복원
        sam.restore()

    # 4) Muon + 보조 옵티마이저 업데이트
    for o in optimizers:
        scaler.step(o)
    scaler.update()
    for s in schedulers:
        s.step()
    return loss.detach(), out.detach(), (loss_adv.detach() if loss_adv is not None else None)

@torch.no_grad()
def loss_on_batch(x, y):
    # 업데이트 직후 같은 배치의 loss (train 모드, BN 통계는 갱신하지 않음)
    set_bn_momentum(model, False)
    loss, _ = forward_loss(x, y)
    set_bn_momentum(model, True)
    return loss

@torch.no_grad()
def evaluate():
    model.eval()
    correct1 = correct5 = total = 0
    loss_sum = 0.0
    for x, y in test_loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=amp_enabled):
            out = model(x)
        loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item()
        top5 = out.topk(5, dim=1).indices
        correct1 += (top5[:, 0] == y).sum().item()
        correct5 += (top5 == y[:, None]).any(dim=1).sum().item()
        total += y.size(0)
    return loss_sum / total, 100 * correct1 / total, 100 * correct5 / total

In [ ]:
import csv

history = {k: [] for k in ["train_loss", "train_acc", "test_loss", "test_acc", "test_top5"]}
best_acc, start_epoch = 0.0, 1
last_path = os.path.join(CFG["save_dir"], "last.pt")
step_log_path = os.path.join(CFG["save_dir"], "step_log.csv")
STEP_COLS = ["step", "epoch", "lr", "loss_w", "loss_w_eps", "loss_after"]
csv_path = os.path.join(CFG["save_dir"], "history.csv")

if CFG["resume"] and os.path.exists(last_path):
    ckpt = torch.load(last_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    for o, sd in zip(optimizers, ckpt["optimizers"]):
        o.load_state_dict(sd)
    for s, sd in zip(schedulers, ckpt["schedulers"]):
        s.load_state_dict(sd)
    scaler.load_state_dict(ckpt["scaler"])
    history, best_acc, start_epoch = ckpt["history"], ckpt["best_acc"], ckpt["epoch"] + 1
    print(f"last.pt에서 이어서 학습: epoch {start_epoch}부터 (best {best_acc:.2f}%)")

for epoch in range(start_epoch, epochs + 1):
    model.train()
    t0 = time.time()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    seen = 0
    step_rows = []
    for it, (x, y) in enumerate(train_loader):
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        lr_now = schedulers[0].get_last_lr()[0]
        loss, out, loss_adv = train_step(x, y)
        if CFG["step_log_every"] and it % CFG["step_log_every"] == 0:
            # loss_w: 업데이트 전 L(w), loss_w_eps: SAM 교란 지점 L(w+ε), loss_after: 업데이트 후 같은 배치의 L(w')
            step_rows.append([(epoch - 1) * steps_per_epoch + it, epoch, lr_now, loss.item(),
                              loss_adv.item() if loss_adv is not None else float("nan"),
                              loss_on_batch(x, y).item()])
        loss_sum += loss.float() * y.size(0)
        correct += (out.argmax(1) == y).sum()
        seen += y.size(0)

    train_loss, train_acc = loss_sum.item() / seen, 100 * correct.item() / seen
    test_loss, test_acc, test_top5 = evaluate()
    for k, v in zip(history, [train_loss, train_acc, test_loss, test_acc, test_top5]):
        history[k].append(v)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({"model": model.state_dict(), "epoch": epoch, "acc": test_acc, "cfg": CFG},
                   os.path.join(CFG["save_dir"], "best.pt"))
    torch.save({"model": model.state_dict(), "optimizers": [o.state_dict() for o in optimizers],
                "schedulers": [s.state_dict() for s in schedulers], "scaler": scaler.state_dict(),
                "history": history, "best_acc": best_acc, "epoch": epoch, "cfg": CFG}, last_path)
    if step_rows:
        new_file = not os.path.exists(step_log_path)
        with open(step_log_path, "a", newline="") as f:
            w = csv.writer(f)
            if new_file:
                w.writerow(STEP_COLS)
            w.writerows(step_rows)
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["epoch", *history])
        w.writerows([i + 1, *(round(history[k][i], 4) for k in history)] for i in range(len(history["train_loss"])))

    print(f"[{epoch:3d}/{epochs}] train loss {train_loss:.3f} acc {train_acc:5.2f}% | "
          f"test loss {test_loss:.3f} top1 {test_acc:5.2f}% top5 {test_top5:5.2f}% | "
          f"best {best_acc:5.2f}% | lr {schedulers[0].get_last_lr()[0]:.4f} | {time.time() - t0:.0f}s")

print(f"\n최고 test top-1 정확도: {best_acc:.2f}%")

## 9. 학습 곡선 (모든 실험 비교)


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

PLOT_RUNS = None

rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, path in enumerate(sorted(glob.glob(os.path.join(RUNS_DIR, "*", "history.csv")))):
    name = os.path.basename(os.path.dirname(path))
    if PLOT_RUNS and name not in PLOT_RUNS:
        continue
    h = pd.read_csv(path)
    color, lw = f"C{i % 10}", (2.4 if name == CFG["exp_name"] else 1.3)
    axes[0].plot(h.epoch, h.test_loss, color=color, lw=lw, label=name)
    axes[0].plot(h.epoch, h.train_loss, color=color, lw=1, ls=":")
    axes[1].plot(h.epoch, h.test_acc, color=color, lw=lw, label=name)
    axes[1].plot(h.epoch, h.train_acc, color=color, lw=1, ls=":")
    best = h.loc[h.test_acc.idxmax()]
    rows.append(dict(experiment=name, epochs=len(h), best_top1=best.test_acc, best_epoch=int(best.epoch),
                     final_top1=h.test_acc.iloc[-1], final_top5=h.test_top5.iloc[-1],
                     min_test_loss=h.test_loss.min()))

axes[0].set_title("Loss (solid: test, dotted: train)"); axes[0].set_xlabel("epoch")
axes[1].set_title("Top-1 accuracy % (solid: test, dotted: train)"); axes[1].set_xlabel("epoch")
axes[1].set_ylim(bottom=max(0, min((r["best_top1"] for r in rows), default=0) - 30))
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

summary = pd.DataFrame(rows).sort_values("best_top1", ascending=False).reset_index(drop=True)
summary.to_csv(os.path.join(ROOT, "summary.csv"), index=False)
summary

## 9-2. 스텝 단위 loss 분석

In [ ]:
ANALYZE_RUN = CFG["exp_name"]   # 다른 실험을 보려면 이름을 넣으세요
SMOOTH = 25                     # 이동 평균 창 크기 (기록 개수 기준)

def load_step_log(name):
    path = os.path.join(RUNS_DIR, name, "step_log.csv")
    if not os.path.exists(path):
        return None
    s = pd.read_csv(path).drop_duplicates("step", keep="last").sort_values("step")
    s["delta"] = s.loss_after - s.loss_w
    s["sharp_gap"] = s.loss_w_eps - s.loss_w
    return s

s = load_step_log(ANALYZE_RUN)
if s is None:
    print(f"{ANALYZE_RUN} 에는 step_log.csv 가 없습니다 (이 기능을 넣기 전 실험이거나 step_log_every=0).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    ax = axes[0]
    ax.plot(s.step, s.loss_w.rolling(SMOOTH, min_periods=1).mean(), label="L(w)")
    if s.loss_w_eps.notna().any():
        ax.plot(s.step, s.loss_w_eps.rolling(SMOOTH, min_periods=1).mean(), label="L(w+eps)")
    ax.plot(s.step, s.loss_after.rolling(SMOOTH, min_periods=1).mean(), label="L(w') after step", ls="--")
    ax.set_title(f"{ANALYZE_RUN}: loss per step (moving avg)"); ax.set_xlabel("step"); ax.legend()

    ax = axes[1]
    up = s.delta > 0
    ax.scatter(s.step[~up], s.delta[~up], s=4, alpha=0.4, color="C0", label="decreased")
    ax.scatter(s.step[up], s.delta[up], s=4, alpha=0.6, color="C3", label="increased")
    ax.plot(s.step, s.delta.rolling(SMOOTH, min_periods=1).mean(), color="k", lw=1.2, label="moving avg")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_title("delta = L(w') - L(w) on same batch"); ax.set_xlabel("step"); ax.legend(markerscale=3)

    ax = axes[2]
    if s.sharp_gap.notna().any():
        ax.plot(s.step, s.sharp_gap.rolling(SMOOTH, min_periods=1).mean(), color="C4")
        ax.set_title("sharpness: L(w+eps) - L(w) (moving avg)")
    else:
        ax.set_title("sharpness: SAM not used in this run")
    ax.set_xlabel("step")
    for a in axes:
        a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"L(w)가 올라간 스텝 비율: 전체 {100 * up.mean():.1f}% "
          f"(기록 {len(s)}개, 평균 Δ {s.delta.mean():+.4f})")

# 모든 실험 비교: epoch별 L(w)가 오른 스텝 비율
plt.figure(figsize=(9, 4))
for i, path in enumerate(sorted(glob.glob(os.path.join(RUNS_DIR, "*", "step_log.csv")))):
    name = os.path.basename(os.path.dirname(path))
    if PLOT_RUNS and name not in PLOT_RUNS:
        continue
    r = load_step_log(name)
    frac = r.groupby("epoch").delta.apply(lambda d: 100 * (d > 0).mean())
    plt.plot(frac.index, frac.values, color=f"C{i % 10}", lw=2.4 if name == ANALYZE_RUN else 1.3, label=name)
plt.title("% of logged steps where L(w) increased"); plt.xlabel("epoch"); plt.ylabel("%")
plt.grid(alpha=0.3); plt.legend(fontsize=9); plt.tight_layout(); plt.show()

## 10. 저장된 최고 모델로 평가

In [ ]:
ckpt = torch.load(os.path.join(CFG["save_dir"], "best.pt"), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
test_loss, test_acc, test_top5 = evaluate()
print(f"epoch {ckpt['epoch']} checkpoint → top-1 {test_acc:.2f}% | top-5 {test_top5:.2f}%")

epoch 186 checkpoint → top-1 79.31% | top-5 94.58%
